# 01.6 · Pandas: groupby, merge, pivot

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/06-pandas-groupby-merge/lesson.ipynb)

Read the [README](README.md) first. The seven-row example comes first so you can check every number against your hand calculation; then the real data.

## 1. Seven orders and a CRM table

In [1]:
import pandas as pd

orders = pd.DataFrame({
    "invoice":  [1001, 1001, 1002, 1003, 1004, 1005, 1006],
    "customer": ["C1", "C1", "C2", "C1", "C3", "C2", "C4"],
    "country":  ["UK", "UK", "DE", "UK", "DE", "DE", "FR"],
    "month":    ["Jan", "Jan", "Jan", "Feb", "Feb", "Feb", "Feb"],
    "revenue":  [50, 30, 80, 40, 60, 20, 90],
})
crm = pd.DataFrame({"customer": ["C1", "C2", "C3", "C5"],
                    "channel":  ["ads", "organic", "ads", "referral"]})
orders

,invoice,customer,country,month,revenue
0,1001,C1,UK,Jan,50
1,1001,C1,UK,Jan,30
2,1002,C2,DE,Jan,80
3,1003,C1,UK,Feb,40
4,1004,C3,DE,Feb,60
5,1005,C2,DE,Feb,20
6,1006,C4,FR,Feb,90


## 2. Groupby: split → apply → combine

`groupby("country")` splits; `["revenue"].sum()` applies; the result is one row per country.

In [2]:
orders.groupby("country")["revenue"].sum()

country
DE    160
FR     90
UK    120
Name: revenue, dtype: int64

UK 120, DE 160, FR 90, as on paper. Now the customer summary. Note **`nunique` on invoice**: C1 has three rows but two orders.

In [3]:
summary = orders.groupby("customer").agg(
    orders=("invoice", "nunique"),
    rows=("invoice", "size"),
    spend=("revenue", "sum"),
)
summary

,orders,rows,spend
customer,,,
C1,2,3,120
C2,2,2,100
C3,1,1,60
C4,1,1,90


## 3. Pivot: a two-key groupby as a grid

`fill_value=0` replaces the NaN where a country had no sales in a month.

In [4]:
orders.pivot_table(index="month", columns="country", values="revenue", aggfunc="sum", fill_value=0)

country,DE,FR,UK
month,,,
Feb,80,90,40
Jan,80,0,80


## 4. Merge: look up each customer in the CRM

`how="left"` keeps every customer in `summary`. C4 is not in the CRM → `NaN`. C5 is in the CRM but never ordered → absent.

In [5]:
merged = pd.merge(summary, crm, on="customer", how="left")
merged

,customer,orders,rows,spend,channel
0,C1,2,3,120,ads
1,C2,2,2,100,organic
2,C3,1,1,60,ads
3,C4,1,1,90,NaN


In [6]:
print("outer merge keeps C5 too:")
pd.merge(summary, crm, on="customer", how="outer")

outer merge keeps C5 too:


,customer,orders,rows,spend,channel
0,C1,2.0,3.0,120.0,ads
1,C2,2.0,2.0,100.0,organic
2,C3,1.0,1.0,60.0,ads
3,C4,1.0,1.0,90.0,NaN
4,C5,NaN,NaN,NaN,referral


In [7]:
merged.groupby("channel")["spend"].mean()

channel
ads         90.0
organic    100.0
Name: spend, dtype: float64

Ads customers average £90, organic £100. Every number matches the README.

## 5. The real data

In [8]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(raw.shape)
raw.head()

(541910, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [9]:
def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df.copy()
    out = out[~out["Invoice"].astype(str).str.startswith("C")]   # drop cancellations
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)]         # drop returns / free items
    out = out.dropna(subset=["Customer ID"])                       # drop guest checkouts
    out = out.drop_duplicates()
    out["Customer ID"] = out["Customer ID"].astype(int)
    out["Revenue"] = out["Quantity"] * out["Price"]
    return out

sales = clean_retail(raw)
print(f"{len(raw):,} rows → {len(sales):,} after cleaning")

541,910 rows → 392,693 after cleaning


### Question 1: revenue per country

In [10]:
by_country = sales.groupby("Country")["Revenue"].sum().sort_values(ascending=False)
share = by_country / by_country.sum()
pd.DataFrame({"revenue": by_country.round(0), "share": share.round(3)}).head(8)

,revenue,share
Country,,
United Kingdom,7285025.0,0.820
Netherlands,285446.0,0.032
EIRE,265262.0,0.030
Germany,228678.0,0.026
France,208952.0,0.024
Australia,138454.0,0.016
Spain,61559.0,0.007
Switzerland,56444.0,0.006


The UK is about 82% of revenue. Everything else is a long tail, which matters when you later ask "is `Country` a useful feature?"

### Question 2: the customer table

This is the most important table in the course. One row per customer. Modules 02–06 build features on it.

In [11]:
customers = sales.groupby("Customer ID").agg(
    orders=("Invoice", "nunique"),
    spend=("Revenue", "sum"),
    products=("StockCode", "nunique"),
    first_order=("InvoiceDate", "min"),
    last_order=("InvoiceDate", "max"),
)
print(customers.shape)
customers.head()

(4338, 5)


,orders,spend,products,first_order,last_order
Customer ID,,,,,
12346,1,77183.60,1,2011-01-18 10:01:00,2011-01-18 10:01:00
12347,7,4310.00,103,2010-12-07 14:57:00,2011-12-07 15:52:00
12348,4,1797.24,22,2010-12-16 19:09:00,2011-09-25 13:13:00
12349,1,1757.55,73,2011-11-21 09:51:00,2011-11-21 09:51:00
12350,1,334.40,17,2011-02-02 16:01:00,2011-02-02 16:01:00


In [12]:
print(f"customers: {len(customers):,}")
print(f"ordered more than once: {(customers['orders'] > 1).sum():,}  ({(customers['orders'] > 1).mean():.0%})")
print(f"ordered more than five times: {(customers['orders'] > 5).sum():,}")
customers["orders"].describe().round(1)

customers: 4,338
ordered more than once: 2,845  (66%)
ordered more than five times: 872


count    4338.0
mean        4.3
std         7.7
min         1.0
25%         1.0
50%         2.0
75%         5.0
max       209.0
Name: orders, dtype: float64

### Question 3: monthly revenue by country, as a grid

In [13]:
sales["month"] = sales["InvoiceDate"].dt.to_period("M")
top5 = by_country.index[:5]
grid = (sales[sales["Country"].isin(top5)]
        .pivot_table(index="month", columns="Country", values="Revenue", aggfunc="sum", fill_value=0)
        .round(0))
grid

Country,EIRE,France,Germany,Netherlands,United Kingdom
month,,,,,
2010-12,8814.0,9616.0,15206.0,8784.0,496477.0
2011-01,21904.0,17740.0,16881.0,26611.0,440876.0
2011-02,10112.0,8516.0,9581.0,23012.0,354618.0
2011-03,21674.0,14585.0,14393.0,22416.0,465784.0
2011-04,7570.0,5530.0,12316.0,2977.0,408733.0
2011-05,15977.0,17606.0,25735.0,29186.0,550359.0
2011-06,19836.0,16079.0,13246.0,26858.0,523776.0
2011-07,40854.0,10000.0,16441.0,26.0,484546.0
2011-08,16958.0,13811.0,19221.0,40328.0,497195.0


November 2011 is the peak in every country: Christmas stock-up. You will forecast this in Module 07.

### Question 4: merging an external table

ShopWave's CRM is not in the dataset, so we generate a sign-up-channel table to practise the merge. The point is the mechanics: key dtype, `how`, and checking the row count.

In [14]:
import numpy as np
rng = np.random.default_rng(1)
crm_real = pd.DataFrame({
    "Customer ID": customers.index,
    "channel": rng.choice(["ads", "organic", "referral", "email"], len(customers), p=[.4, .3, .15, .15]),
}).sample(frac=0.9, random_state=1)        # 10% of customers are missing from the CRM, as in real life

before = len(customers)
cust_ch = pd.merge(customers, crm_real, on="Customer ID", how="left", validate="one_to_one")
print(f"rows before {before:,}, after {len(cust_ch):,}")
print("customers with no channel:", cust_ch["channel"].isna().sum())
cust_ch.groupby("channel")["spend"].agg(["count", "mean", "median"]).round(0)

rows before 4,338, after 4,338
customers with no channel: 434


,count,mean,median
channel,,,
ads,1586,2152.0,683.0
email,590,1723.0,682.0
organic,1166,2227.0,634.0
referral,562,1616.0,692.0


`validate="one_to_one"` would raise an error if either side had duplicate keys, which is exactly the silent row-multiplication bug you want to catch. (The channel means are all similar because the channels were random; with a real CRM they would not be.)

## Quiz answers

1. Split (by key) → apply (a function per group) → combine (into one table).
2. `nunique` on the invoice column. `count` counts rows with a non-missing value, `size` counts rows.
3. Left merge: the row is kept with `NaN` in the right-hand columns. Inner merge: the row is dropped.
4. The key is not unique on the right-hand table, so each left row matched several right rows. Deduplicate the right table or use `validate=`.
5. When some (index, column) pairs occur more than once and must be aggregated. `pivot` only reshapes and errors on duplicates.

**Next:** [01.7 Plotting](../07-plotting/)